# <font color="#418FDE" size="6.5" uppercase>**Manuell vorverarbeiten**</font>

>Last update: 20260825.
    
By the end of this Lecture, you will be able to:
- Bereinigen fehlende, doppelte, unplausible und uneinheitliche Datenwerte. 
- Transformieren numerische, kategoriale, Text- und Datumsmerkmale manuell. 
- Entwerfen eine überprüfbare Vorverarbeitungsfunktion mit Kontrolltabellen. 


## **1. Daten gezielt bereinigen**

### **1.1. Fehlwerte sinnvoll ersetzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_01.jpg?v=1787631965" width="250">



>* Fehlwerte zuerst fachlich verstehen
>* Zufällige und systematische Muster prüfen

>* Ersetzung passend zum Merkmal wählen
>* Pauschale Ersetzungen fachlich kritisch prüfen

>* Ersetzungen sparsam dokumentieren und markieren
>* Gruppenspezifisch prüfen oder Fehlwerte belassen



In [ ]:
#@title Python-Code - Fehlwerte sinnvoll ersetzen

# Fehlwerte werden hier bewusst und nachvollziehbar ersetzt.
# Median und Kategorie erhalten wichtige Datenstruktur.
# Kontrolltabellen zeigen die Wirkung der Bereinigung.

import pandas as pd
import matplotlib.pyplot as plt

# Ein kleiner Datensatz enthält typische fehlende Angaben.
data = pd.DataFrame(
    {
        "customer": ["A", "B", "C", "D", "E", "F"],
        "age": [29, None, 41, 38, None, 52],
        "region": ["Nord", "Süd", None, "Nord", "Süd", None],
    }
)

# Wir merken uns, welche Alterswerte ursprünglich fehlten.
cleaned = data.copy()
cleaned["age_was_missing"] = cleaned["age"].isna()

# Der Median ist robust gegenüber einzelnen extremen Werten.
age_median = cleaned["age"].median()
cleaned["age"] = cleaned["age"].fillna(age_median)

# Fehlende Kategorien bekommen eine eigene, ehrliche Bezeichnung.
cleaned["region"] = cleaned["region"].fillna("unbekannt")

# Eine Kontrolltabelle macht die Ersetzungen überprüfbar.
missing_before = data.isna().sum()
missing_after = cleaned.isna().sum()
control = pd.DataFrame(
    {"vorher": missing_before, "nachher": missing_after}
)

print("Kontrolle der Fehlwerte je Spalte:")
print(control.to_string())
print("Eingesetzter Altersmedian:", round(age_median, 1))
print("Markierte ersetzte Alterswerte:", int(cleaned["age_was_missing"].sum()))

# Die Grafik zeigt Originalwerte und ersetzte Werte getrennt.
fig, ax = plt.subplots(figsize=(7, 4))
colors = cleaned["age_was_missing"].map({False: "steelblue", True: "orange"})
ax.bar(cleaned["customer"], cleaned["age"], color=colors)

ax.set_title("Ersetzte Alterswerte bleiben sichtbar")
ax.set_xlabel("Kundin oder Kunde")
ax.set_ylabel("Alter in Jahren")
ax.legend(["blau: original, orange: ersetzt"], loc="upper left")
plt.show()



### **1.2. Duplikate und Datentypen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_02.jpg?v=1787631966" width="250">



>* Duplikate können auch leicht abweichen
>* Zeilenbedeutung entscheidet über Bereinigung

>* Duplikate erst prüfen, dann bereinigen
>* Schlüssel nutzen und Entscheidungen dokumentieren

>* Datentypen fachlich passend vereinheitlichen
>* Saubere Typisierung macht Prüfungen zuverlässiger



In [ ]:
#@title Python-Code - Duplikate und Datentypen

# Dieses Beispiel bereinigt Duplikate und Datentypen.
# Fachliche Schlüssel verhindern falsches Löschen echter Fälle.
# Am Ende stehen saubere Kontrollzahlen bereit.

import pandas as pd

# Kleine Rohdaten zeigen typische Importprobleme.
raw_data = pd.DataFrame(
    {
        "order_id": [101, 101, 102, 103, 103, 104],
        "customer": ["Mia", "Mia ", "Noah", "Lea", "Lea", "Omar"],
        "order_date": ["01.03.2024", "01.03.2024", "02.03.2024", "03.03.2024", "03.03.2024", "04.03.2024"],
        "amount_eur": ["19,90 €", "19,90 €", "5,50 €", "12,00 €", "12,00 €", "8,00 €"],
    }
)

# Eine Kopie schützt die ursprünglichen Rohdaten.
clean_data = raw_data.copy()

# Textwerte werden vereinheitlicht, bevor Duplikate geprüft werden.
clean_data["customer"] = clean_data["customer"].str.strip()
clean_data["amount_eur"] = clean_data["amount_eur"].str.replace(" €", "", regex=False)
clean_data["amount_eur"] = clean_data["amount_eur"].str.replace(",", ".", regex=False)

# Fachlich passende Datentypen machen Vergleiche zuverlässiger.
clean_data["order_date"] = pd.to_datetime(clean_data["order_date"], format="%d.%m.%Y")
clean_data["amount_eur"] = clean_data["amount_eur"].astype(float)

# Der Schlüssel beschreibt hier eine Bestellung.
duplicate_mask = clean_data.duplicated(subset=["order_id"], keep="first")
clean_data = clean_data.loc[~duplicate_mask].reset_index(drop=True)

# Eine Kontrolltabelle dokumentiert die Bereinigung nachvollziehbar.
control_table = pd.DataFrame(
    {
        "Kennzahl": ["Zeilen vorher", "Duplikate entfernt", "Zeilen nachher"],
        "Wert": [len(raw_data), int(duplicate_mask.sum()), len(clean_data)],
    }
)

# Kurze Ausgaben zeigen Ergebnis und Datentypen.
print(control_table.to_string(index=False))
print("Datentyp Betrag:", str(clean_data["amount_eur"].dtype))
print("Datentyp Datum:", str(clean_data["order_date"].dtype))
print("Bereinigte Bestellungen:", clean_data["order_id"].tolist())



### **1.3. Ausreißer markieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_01_03.jpg?v=1787631968" width="250">



>* Ausreißer beeinflussen Analysen und Modelle stark
>* Erst markieren, dann fachlich entscheiden

>* Statistik immer mit Fachwissen verbinden
>* Markierungsregeln nachvollziehbar dokumentieren

>* Ausreißer immer im passenden Gruppenkontext prüfen
>* Entscheidungen dokumentieren und Datenqualität sichern



In [ ]:
#@title Python-Code - Ausreißer markieren

# Dieses Beispiel markiert Ausreißer in Rechnungsbeträgen.
# Die IQR-Regel trennt Erkennen von Löschen.
# Am Ende sehen wir markierte auffällige Werte.

import pandas as pd
import matplotlib.pyplot as plt

# Wir erstellen kleine Beispieldaten mit plausiblen und extremen Beträgen.
data = pd.DataFrame(
    {
        "invoice_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
        "amount_eur": [48, 52, 55, 49, 51, 53, 50, 54, 300, 5],
    }
)

# Eine einfache Prüfung schützt vor unerwartet leeren Daten.
if len(data) == 0:
    raise ValueError("Der Beispieldatensatz darf nicht leer sein.")

# Quartile beschreiben den mittleren Bereich der Verteilung.
q1 = data["amount_eur"].quantile(0.25)
q3 = data["amount_eur"].quantile(0.75)
iqr = q3 - q1

# Die IQR-Grenzen markieren ungewöhnlich niedrige oder hohe Werte.
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

# Wir markieren Ausreißer, ohne die Originalwerte zu löschen.
data["is_outlier"] = (
    (data["amount_eur"] < lower_limit)
    | (data["amount_eur"] > upper_limit)
)

# Eine Kontrolltabelle macht die Bereinigungsregel nachvollziehbar.
control_table = pd.DataFrame(
    {
        "Kennzahl": ["Q1", "Q3", "IQR", "untere Grenze", "obere Grenze"],
        "Wert": [q1, q3, iqr, lower_limit, upper_limit],
    }
)

# Wir zeigen nur die wichtigsten Kontrollwerte kompakt an.
print("Kontrolltabelle zur IQR-Regel:")
print(control_table.round(2).to_string(index=False))

# Die markierten Zeilen zeigen, welche Werte geprüft werden sollten.
flagged = data.loc[data["is_outlier"], ["invoice_id", "amount_eur"]]
print("Markierte Ausreißer:")
print(flagged.to_string(index=False))

# Farben machen normale und auffällige Rechnungen unterscheidbar.
colors = data["is_outlier"].map({False: "steelblue", True: "crimson"})

# Ein Streudiagramm zeigt die Markierung im Wertebereich.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(data["invoice_id"], data["amount_eur"], c=colors, s=80)
ax.axhline(lower_limit, color="gray", linestyle="--", label="IQR-Grenzen")
ax.axhline(upper_limit, color="gray", linestyle="--")

# Achsen und Titel erklären die sichtbare Entscheidung.
ax.set_title("Ausreißer markieren, nicht sofort löschen")
ax.set_xlabel("Rechnungsnummer")
ax.set_ylabel("Rechnungsbetrag in Euro")
ax.legend()

plt.show()



## **2. Merkmale manuell transformieren**

### **2.1. Einheiten und Texte**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_01.jpg?v=1787631972" width="250">



>* Einheiten und Schreibweisen konsequent vereinheitlichen
>* Transformationen fachlich prüfen und dokumentieren

>* Bedeutung und Darstellung von Einheiten trennen
>* Zielgröße festlegen und Annahmen dokumentieren

>* Textvarianten gezielt vereinheitlichen und bereinigen
>* Bedeutungsunterschiede fachlich prüfen und erhalten



In [ ]:
#@title Python-Code - Einheiten und Texte

# Wir vereinheitlichen Einheiten und Textwerte.
# Rohdaten enthalten absichtlich gemischte Schreibweisen.
# Das Ergebnis zeigt vergleichbare, bereinigte Merkmale.

import pandas as pd

# Diese kleine Tabelle simuliert typische Rohdaten.
raw_data = pd.DataFrame(
    {
        "product": ["Apfel", "apfel ", "APPLE", "Banane", "banane"],
        "amount_text": ["500 g", "0.5 kg", "250 g", "1 kg", "750 g"],
        "country": ["DE", "Deutschland", "Germany", "AT", "Österreich"],
    }
)

# Diese Funktion wandelt Gewichtsangaben einheitlich in Kilogramm um.
def parse_weight_kg(value):
    text = str(value).strip().lower().replace(",", ".")
    number_text = text.split()[0]
    number = float(number_text)

    if "kg" in text:
        return number
    if "g" in text:
        return number / 1000
    raise ValueError("Unbekannte Gewichtseinheit")

# Diese Zuordnung fasst gleichbedeutende Ländertexte zusammen.
country_map = {
    "de": "Deutschland",
    "deutschland": "Deutschland",
    "germany": "Deutschland",
    "at": "Österreich",
    "österreich": "Österreich",
}

# Wir arbeiten auf einer Kopie, damit Rohdaten erhalten bleiben.
clean_data = raw_data.copy()
clean_data["product_clean"] = clean_data["product"].str.strip().str.lower()
clean_data["weight_kg"] = clean_data["amount_text"].apply(parse_weight_kg)

# Auch Ländertexte werden zuerst technisch normalisiert.
country_key = clean_data["country"].str.strip().str.lower()
clean_data["country_clean"] = country_key.map(country_map)

# Eine einfache Kontrolle prüft, ob alle Länder erkannt wurden.
missing_countries = clean_data["country_clean"].isna().sum()
print(f"Unbekannte Länder nach Bereinigung: {missing_countries}")

# Diese Kontrolltabelle zeigt Rohwert und bereinigte Zielmerkmale.
control_table = clean_data[
    ["amount_text", "weight_kg", "product_clean", "country_clean"]
]
print(control_table.to_string(index=False))



### **2.2. Merkmale skalieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_02.jpg?v=1787631974" width="250">



>* Skalierung macht unterschiedliche Merkmale vergleichbar
>* Sie verhindert Verzerrungen und zeigt Muster

>* Gemeinsame Skalen machen Werte vergleichbar
>* Kontext prüfen und Transformationen dokumentieren

>* Ausreißer und Datenaufteilung sorgfältig prüfen
>* Skalierung reproduzierbar dokumentieren und kontrollieren



In [ ]:
#@title Python-Code - Merkmale skalieren

# Dieses Beispiel zeigt manuelle Skalierung numerischer Merkmale.
# Min-Max-Skalierung und Standardisierung werden direkt verglichen.
# Kontrollwerte machen die Transformation nachvollziehbar und prüfbar.

import pandas as pd
import matplotlib.pyplot as plt

# Kleine Beispieldaten bleiben vollständig im Arbeitsspeicher.
data = pd.DataFrame(
    {
        "income_eur": [22000, 35000, 48000, 61000, 90000],
        "age_years": [22, 31, 45, 52, 64],
        "rating_points": [2, 3, 4, 4, 5],
    }
)

# Wir prüfen zuerst die erwartete Tabellenform.
if data.shape != (5, 3):
    raise ValueError("Die Beispieltabelle hat nicht die erwartete Form.")

# Min-Max-Skalierung bringt Werte in den Bereich null bis eins.
min_values = data.min()
max_values = data.max()
value_ranges = max_values - min_values

# Diese Prüfung verhindert eine Division durch null.
if (value_ranges == 0).any():
    raise ValueError("Mindestens ein Merkmal hat keine Streuung.")

minmax_scaled = (data - min_values) / value_ranges
minmax_scaled = minmax_scaled.add_suffix("_minmax")

# Standardisierung beschreibt Abweichungen vom Mittelwert.
mean_values = data.mean()
std_values = data.std(ddof=0)

# Auch hier darf die Streuung nicht null sein.
if (std_values == 0).any():
    raise ValueError("Mindestens ein Merkmal hat Standardabweichung null.")

standard_scaled = (data - mean_values) / std_values
standard_scaled = standard_scaled.add_suffix("_standard")

# Eine Kontrolltabelle zeigt typische Kennzahlen nach der Skalierung.
control_table = pd.DataFrame(
    {
        "minmax_min": minmax_scaled.min(),
        "minmax_max": minmax_scaled.max(),
        "standard_mean": standard_scaled.mean(),
        "standard_std": standard_scaled.std(ddof=0),
    }
)

# Kurze Ausgabe vermeidet unübersichtliche Tabellen.
print("Kontrolltabelle gerundet:")
print(control_table.round(2).to_string())

# Für die Grafik vergleichen wir ein Merkmal vor und nachher.
plot_data = pd.DataFrame(
    {
        "original": data["income_eur"],
        "minmax": minmax_scaled["income_eur_minmax"],
        "standard": standard_scaled["income_eur_standard"],
    }
)

# Ein einzelnes Diagramm zeigt die veränderten Zahlenbereiche.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(plot_data.index + 1, plot_data["original"], marker="o", label="Original")
ax.plot(plot_data.index + 1, plot_data["minmax"], marker="o", label="Min-Max")
ax.plot(plot_data.index + 1, plot_data["standard"], marker="o", label="Standardisiert")

# Achsen und Legende machen die Darstellung lesbar.
ax.set_title("Einkommen vor und nach manueller Skalierung")
ax.set_xlabel("Beobachtung")
ax.set_ylabel("Wert in jeweiliger Skala")
ax.legend()

plt.show()



### **2.3. Binning und Log**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_02_03.jpg?v=1787631970" width="250">



>* Binning gruppiert Zahlen für robustere Muster.
>* Klassen fachlich passend und überprüfbar wählen.

>* Klassengrenzen eindeutig setzen und Randfälle klären
>* Verteilung prüfen, Schwellen begründen und dokumentieren

>* Log glättet rechtsschiefe numerische Verteilungen
>* Nullen, negative Werte und Interpretation prüfen



In [ ]:
#@title Python-Code - Binning und Log

# Dieses Beispiel zeigt Binning und Log-Transformation.
# Wir nutzen kleine synthetische Umsatzdaten.
# Die Ausgabe vergleicht Originalwerte und Transformationen.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Die Daten enthalten typische rechtsschiefe Monatsumsätze.
sales_eur = np.array([80, 120, 150, 220, 300, 450, 700, 1200, 2500, 9000])

# Eine einfache Prüfung verhindert ungültige Logarithmen.
if np.any(sales_eur < 0):
    raise ValueError("Umsätze dürfen hier nicht negativ sein.")

# Binning ordnet Zahlen in fachlich benannte Gruppen ein.
bin_edges = [0, 500, 2000, np.inf]
bin_labels = ["niedrig", "mittel", "hoch"]

# Die rechte Grenze gehört jeweils zur aktuellen Klasse.
sales_bins = pd.cut(
    sales_eur, bins=bin_edges, labels=bin_labels, include_lowest=True
)

# Log1p funktioniert auch bei null und komprimiert große Werte.
log_sales = np.log1p(sales_eur)

# Eine kleine Kontrolltabelle macht die Transformation überprüfbar.
result = pd.DataFrame(
    {"umsatz_eur": sales_eur, "klasse": sales_bins, "log1p": log_sales}
)

# Wir zeigen nur wenige Zeilen, damit die Ausgabe übersichtlich bleibt.
print("Kontrolltabelle der ersten fünf Kunden:")
print(result.head(5).round({"log1p": 2}).to_string(index=False))

# Die Klassenhäufigkeiten zeigen, ob Bins sinnvoll gefüllt sind.
counts = result["klasse"].value_counts(sort=False)
print("Klassenhäufigkeiten:", counts.to_dict())

# Das Streudiagramm zeigt die Kompression großer Werte.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(result["umsatz_eur"], result["log1p"], s=70, color="tab:blue")

# Achsen und Titel benennen Originalskala und Log-Skala.
ax.set_title("Log-Transformation komprimiert große Umsätze")
ax.set_xlabel("Umsatz in Euro")
ax.set_ylabel("log1p(Umsatz)")
ax.grid(True, alpha=0.3)

plt.show()



## **3. Merkmale überprüfbar erzeugen**

### **3.1. Kategorien nachvollziehbar kodieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_01.jpg?v=1787631975" width="250">



>* Kategorien nachvollziehbar und reproduzierbar kodieren
>* Ausprägungen prüfen, vereinheitlichen und dokumentieren

>* Nominale und ordinale Kategorien getrennt behandeln
>* Kodierung dokumentieren und per Kontrolltabelle prüfen

>* Unbekannte Kategorien sichtbar und kontrolliert behandeln
>* Kodierung dokumentieren, prüfen und Verzerrungen vermeiden



In [ ]:
#@title Python-Code - Kategorien nachvollziehbar kodieren

# Dieses Beispiel kodiert Kategorien nachvollziehbar.
# Kontrolltabellen machen jede Zuordnung prüfbar.
# Unbekannte Werte werden sichtbar behandelt.

import pandas as pd

# Kleine Rohdaten zeigen typische uneinheitliche Schreibweisen.
raw_data = pd.DataFrame(
    {
        "payment_method": ["Kreditkarte", "kreditkarte", "PayPal", "Rechnung", "Bar"],
        "satisfaction": ["hoch", "mittel", "niedrig", "hoch", "unbekannt"],
    }
)

# Fachliche Regeln standardisieren nominale und ordinale Kategorien.
payment_map = {
    "kreditkarte": "Kreditkarte",
    "paypal": "PayPal",
    "rechnung": "Rechnung",
}

satisfaction_map = {"niedrig": 1, "mittel": 2, "hoch": 3}

# Die Funktion erzeugt kodierte Merkmale und Kontrolltabellen.
def preprocess_categories(data):
    result = data.copy()
    result["payment_clean"] = result["payment_method"].str.lower().map(payment_map)

    result["payment_clean"] = result["payment_clean"].fillna("Unbekannt")
    result["satisfaction_code"] = result["satisfaction"].map(satisfaction_map)

    payment_dummies = pd.get_dummies(result["payment_clean"], prefix="payment")
    result = pd.concat([result, payment_dummies], axis=1)

    payment_check = result.groupby(
        ["payment_method", "payment_clean"], dropna=False
    ).size().reset_index(name="rows")

    satisfaction_check = result.groupby(
        ["satisfaction", "satisfaction_code"], dropna=False
    ).size().reset_index(name="rows")

    return result, payment_check, satisfaction_check

# Die Vorverarbeitung läuft reproduzierbar auf denselben Regeln.
processed_data, payment_check, satisfaction_check = preprocess_categories(raw_data)

# Eine einfache Prüfung schützt vor still verlorenen Zeilen.
if len(processed_data) != len(raw_data):
    raise ValueError("Die Zeilenzahl hat sich unerwartet verändert.")

# Die Ausgabe bleibt kurz und zeigt die wichtigsten Kontrollen.
print("Kodierte Spalten:", list(processed_data.columns[-4:]))
print("Unbekannte Zahlungsarten:", int((processed_data["payment_clean"] == "Unbekannt").sum()))
print("Kontrolle Zahlungsart:")
print(payment_check.head(5).to_string(index=False))
print("Kontrolle Zufriedenheit:")
print(satisfaction_check.head(5).to_string(index=False))



### **3.2. Interaktionen gezielt prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_02.jpg?v=1787631979" width="250">



>* Interaktionen zeigen abhängige Bedeutungen von Merkmalen
>* Fachlich begründen und gezielt auf Plausibilität prüfen

>* Neue Merkmale mit Ausgangswerten vergleichen
>* Randfälle und Gruppen per Kontrolltabelle prüfen

>* Interaktionen fachlich begründen und dokumentieren
>* Kontrolltabellen zeigen typische und auffällige Fälle



In [ ]:
#@title Python-Code - Interaktionen gezielt prüfen

# Dieses Beispiel prüft ein Interaktionsmerkmal gezielt.
# Kontrolltabellen zeigen Ausgangswerte und abgeleitete Werte.
# Auffällige Fälle werden nachvollziehbar markiert.

import pandas as pd
import matplotlib.pyplot as plt

# Kleine Beispieldaten bleiben vollständig im Arbeitsspeicher.
data = pd.DataFrame(
    {
        "apartment_id": [101, 102, 103, 104, 105, 106, 107, 108],
        "area_sqm": [45, 80, 120, 30, 200, 55, 95, 10],
        "rooms": [2, 4, 3, 0, 5, 1, 8, 2],
        "region": ["Nord", "Nord", "Süd", "Süd", "West", "West", "Nord", "Süd"],
    }
)

# Eine einfache Prüfung schützt vor Division durch null.
if (data["rooms"] < 1).any():
    print("Prüfung: Mindestens eine Wohnung hat keine gültige Zimmerzahl.")

# Das Interaktionsmerkmal verbindet Fläche und Zimmerzahl.
clean_rooms = data["rooms"].where(data["rooms"] >= 1)
data["sqm_per_room"] = data["area_sqm"] / clean_rooms

data["check_flag"] = "plausibel"
data.loc[data["rooms"] < 1, "check_flag"] = "Zimmerzahl prüfen"
data.loc[data["sqm_per_room"] < 12, "check_flag"] = "sehr klein prüfen"
data.loc[data["sqm_per_room"] > 45, "check_flag"] = "sehr groß prüfen"

# Die Kontrolltabelle zeigt nur relevante Prüfspalten.
control_table = data[
    ["apartment_id", "area_sqm", "rooms", "sqm_per_room", "check_flag"]
].copy()

control_table["sqm_per_room"] = control_table["sqm_per_room"].round(1)
print("Kontrolltabelle für auffällige Interaktionen:")
print(control_table[control_table["check_flag"] != "plausibel"].head(5).to_string(index=False))

# Gruppenwerte helfen, Muster statt Einzelfälle zu erkennen.
region_summary = data.groupby("region")["sqm_per_room"].median().round(1)
print("Median Quadratmeter pro Zimmer nach Region:")
print(region_summary.to_string())

# Die Grafik vergleicht Ausgangswerte und Interaktionsprüfung.
fig, ax = plt.subplots(figsize=(7, 4))
colors = data["check_flag"].map(lambda value: "tab:red" if value != "plausibel" else "tab:blue")
ax.scatter(data["rooms"], data["area_sqm"], c=colors, s=80)
ax.set_title("Wohnfläche und Zimmerzahl gezielt prüfen")
ax.set_xlabel("Zimmerzahl")
ax.set_ylabel("Wohnfläche in Quadratmetern")
plt.show()



### **3.3. Vorverarbeitung praktisch prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_04/Lecture_B/image_03_03.jpg?v=1787631977" width="250">



>* Merkmale fachlich und statistisch kontrollieren
>* Kontrolltabellen machen Veränderungen nachvollziehbar

>* Kombinierte Schritte können unerwartete Verzerrungen erzeugen
>* Verteilungen, Namen und Änderungen sorgfältig dokumentieren

>* Ergebnisse reproduzierbar, erklärbar und vergleichbar machen
>* Kontrollen sichern Qualität und verantwortliche Entscheidungen



In [ ]:
#@title Python-Code - Vorverarbeitung praktisch prüfen

# Wir prüfen Vorverarbeitung mit kleinen Kontrolltabellen.
# Jede Regel erzeugt nachvollziehbare neue Merkmale.
# Am Ende vergleichen wir Rohdaten und Ergebnis.

import pandas as pd
import matplotlib.pyplot as plt

# Kleine Rohdaten zeigen typische Qualitätsprobleme.
raw_data = pd.DataFrame(
    {
        "customer_id": [1, 2, 2, 3, 4, 5],
        "country": ["Deutschland", "DE", "Germany", "France", "FR", None],
        "birth_year": [1988, 2030, 1988, 1975, None, 1890],
        "delivery_days": [2, None, None, 9, 40, 3],
    }
)

# Die Funktion verändert eine Kopie und sammelt Prüfwerte.
def preprocess_customers(input_data):
    data = input_data.copy()
    checks = []

    # Doppelte Kundennummern werden vor der Bereinigung gezählt.
    duplicate_count = int(data.duplicated("customer_id").sum())
    data = data.drop_duplicates("customer_id", keep="first")

    # Länderangaben werden auf wenige klare Kategorien vereinheitlicht.
    country_map = {"Deutschland": "DE", "Germany": "DE", "France": "FR"}
    data["country_clean"] = data["country"].replace(country_map).fillna("Unbekannt")

    # Unplausible Geburtsjahre werden als fehlend behandelt.
    valid_year = data["birth_year"].between(1920, 2024)
    data["age"] = 2024 - data["birth_year"].where(valid_year)

    # Fehlende Lieferzeiten erhalten einen dokumentierten Ersatzwert.
    missing_delivery = int(data["delivery_days"].isna().sum())
    data["delivery_days_clean"] = data["delivery_days"].fillna(7).clip(0, 30)

    # Kontrolltabelle fasst die wichtigsten Eingriffe zusammen.
    checks.append(["Doppelte Zeilen", duplicate_count])
    checks.append(["Unbekannte Länder", int((data["country_clean"] == "Unbekannt").sum())])
    checks.append(["Ungültige Geburtsjahre", int(data["age"].isna().sum())])
    checks.append(["Ersetzte Lieferzeiten", missing_delivery])

    check_table = pd.DataFrame(checks, columns=["Prüfung", "Anzahl"])
    return data, check_table

# Die Vorverarbeitung liefert Daten und Kontrolltabelle gemeinsam.
clean_data, check_table = preprocess_customers(raw_data)

# Eine einfache Prüfung schützt vor stillen Strukturfehlern.
if len(clean_data) != clean_data["customer_id"].nunique():
    raise ValueError("Die Kundennummern sind nach der Bereinigung nicht eindeutig.")

# Kurze Ausgaben zeigen, was geprüft wurde.
print("Kontrolltabelle der Vorverarbeitung:")
print(check_table.to_string(index=False))
print("Bereinigte Vorschau:")
print(clean_data[["customer_id", "country_clean", "age"]].head(3).to_string(index=False))

# Das Diagramm macht die Ländervereinheitlichung sichtbar.
country_counts = clean_data["country_clean"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(country_counts.index, country_counts.values, color="steelblue")
ax.set_title("Bereinigte Länderkategorien")
ax.set_xlabel("Land")
ax.set_ylabel("Anzahl Kunden")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Manuell vorverarbeiten**</font>


In this lecture, you learned to:
- Bereinigen fehlende, doppelte, unplausible und uneinheitliche Datenwerte. 
- Transformieren numerische, kategoriale, Text- und Datumsmerkmale manuell. 
- Entwerfen eine überprüfbare Vorverarbeitungsfunktion mit Kontrolltabellen. 

In the next Module (Module 5), we will go over 'Bilder und Signale'